In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
yashdave378567567_acrade_stenosis_path = kagglehub.dataset_download('yashdave378567567/acrade-stenosis')

print('Data source import complete.')


In [ ]:
import os

# Pointing to the deep nested folder from your screenshot
# Note the spelling 'acrade' to match your dataset name
MESSY_PATH = "/kaggle/input/acrade-stenosis/Users/SAP/Downloads/stenosis"

if os.path.exists(MESSY_PATH):
    print("✅ Path found!")
    print("Contents:", os.listdir(MESSY_PATH))
else:
    print("❌ Path not found. Let's look for it...")
    # This helper will find where 'stenosis' is hiding
    for root, dirs, files in os.walk("/kaggle/input"):
        if "stenosis" in dirs:
            print(f"Found it here: {os.path.join(root, 'stenosis')}")

In [ ]:
import json
import os
import shutil

# ================= ADJUSTED PATHS =================
# We point specifically to your messy upload path
INPUT_ROOT = "/kaggle/input/acrade-stenosis/Users/SAP/Downloads/stenosis"
OUTPUT_ROOT = "/kaggle/working/yolo_dataset"
# =================================================

def clean_and_convert(subset):
    # Paths based on your screenshot structure
    json_file = f"{INPUT_ROOT}/{subset}/annotations/{subset}.json"
    img_dir = f"{INPUT_ROOT}/{subset}/images"

    # Clean output paths
    out_img_dir = f"{OUTPUT_ROOT}/{subset}/images"
    out_lbl_dir = f"{OUTPUT_ROOT}/{subset}/labels"

    # Create folders
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    print(f"🚀 Fixing {subset} data...")

    if not os.path.exists(json_file):
        print(f"⚠️ Could not find {json_file}")
        return

    # Load and Convert
    with open(json_file, 'r') as f:
        data = json.load(f)

    images_info = {img['id']: img for img in data['images']}

    count = 0
    for ann in data['annotations']:
        img_info = images_info.get(ann['image_id'])
        if not img_info: continue

        file_name = img_info['file_name']
        img_w = img_info['width']
        img_h = img_info['height']

        # Normalize BBox
        x, y, w, h = ann['bbox']
        x_c = (x + w/2) / img_w
        y_c = (y + h/2) / img_h
        w_n = w / img_w
        h_n = h / img_h

        # Save Label
        txt_name = os.path.splitext(file_name)[0] + ".txt"
        with open(os.path.join(out_lbl_dir, txt_name), 'a') as f:
            f.write(f"0 {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")

        # Copy Image
        src = os.path.join(img_dir, file_name)
        dst = os.path.join(out_img_dir, file_name)
        if not os.path.exists(dst):
            shutil.copy(src, dst)

        count += 1
    print(f"✅ {subset} fixed: {count} images ready.")

# Run for both
clean_and_convert('train')
clean_and_convert('val')

In [ ]:
!pip install ultralytics
import ultralytics
from ultralytics import YOLO
import os

print("------------------------------------------------")
print("🚀 STEP 1: Setting up configuration...")
print("------------------------------------------------")

# define the config path
yaml_path = '/kaggle/working/data.yaml'

# Create the YAML file
yaml_content = """
path: /kaggle/working/yolo_dataset
train: train/images
val: val/images
nc: 1
names: ['stenosis']
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✅ Config saved to {yaml_path}")

print("------------------------------------------------")
print("🧠 STEP 2: Loading the Model...")
print("------------------------------------------------")

# Load the model
model = YOLO('yolov8n.pt')
print("✅ Model loaded successfully.")

print("------------------------------------------------")
print("🔥 STEP 3: STARTING TRAINING")
print("⚠️ Note: The first Epoch takes longest to start!")
print("------------------------------------------------")

# Train with verbose=True to ensure output
results = model.train(
    data=yaml_path,
    epochs=20,
    imgsz=512,
    batch=16,
    verbose=True,  # This forces the progress bar
    name='cad_model'
)

print("------------------------------------------------")
print("🏆 TRAINING COMPLETE!")
print("------------------------------------------------")